In [4]:
import random
import copy

COLORS = ['Red', 'Blue', 'Green', 'Yellow'] 
VALUES = [str(i) for i in range(10)] + ['Skip'] 

class Card: 
    def __init__(self, color, value):
        self.color = color
        self.value = value

    def __repr__(self):
        return f"{self.color} {self.value}"

#  Deck Generation 
def generate_deck():
    deck = [Card(color, value) for color in COLORS for value in VALUES]
    random.shuffle(deck)
    return deck

def get_valid_moves(hand, top_card):
    valid_moves = []
    for card in hand:
        
        # Rule 1: A player may play a card if it matches the color OR number 
        if card.color == top_card.color or card.value == top_card.value:
            valid_moves.append(card)
            
    return valid_moves

# State Transition 
def apply_move(state, player_key, move):
    new_state = copy.deepcopy(state)
    
    # Rule 2: If no valid card exists, player must draw 1 card 
    if move == "Draw":
        if len(new_state['deck']) > 0:
            drawn_card = new_state['deck'].pop()
            new_state[player_key].append(drawn_card)
    else:
        for i, card in enumerate(new_state[player_key]):
            if card.color == move.color and card.value == move.value:
                new_state[player_key].pop(i)
                break
        new_state['top_card'] = move
        
    return new_state

def simulate_random_game(initial_state):
    current_state = copy.deepcopy(initial_state)
    players = ['Player1', 'Player2', 'Ai'] 
    turn_index = 0
    
    print(f"starting game! top card: {current_state['top_card']}")
    
    while True:
        current_player = players[turn_index % 3]
        hand = current_state[current_player]
        
        # Rule 4: player wins when cards = 0 
        if len(hand) == 0:
            print(f"\ngame over! {current_player} wins!")
            break 
            
        print(f"\n--- {current_player}'s turn ---")
        print(f"hand: {hand}")
        
        if current_player == 'Player1':
            score = evaluate_state(current_state, current_player, "defensive")
        elif current_player == 'Player2':
            score = evaluate_state(current_state, current_player, "offensive")
        else:
            score = evaluate_state(current_state, current_player, "defensive")
            
        print(f"current score: {score}")
        
        valid_moves = get_valid_moves(hand, current_state['top_card'])
        
        if valid_moves:
            chosen_move = random.choice(valid_moves)
            print(f"action: plays {chosen_move}")
        else:
            chosen_move = "Draw"
            print("action: draws a card")
            
        current_state = apply_move(current_state, current_player, chosen_move)
        
        # Rule 3: if player plays skip, next player's turn is skipped 
        if chosen_move != "Draw" and chosen_move.value == 'Skip':
            print(f"skip! the next player loses their turn.")
            turn_index += 2 
        else:
            turn_index += 1 
            
        if len(current_state['deck']) == 0:
            print("\ndraw deck is empty. it's a tie!")
            break

def evaluate_state(st, p, strat="baseline"):
    ops = [x for x in ['Player1', 'Player2', 'Ai'] if x != p]
    
    my_c = len(st[p])
    op_c = (len(st[ops[0]]) + len(st[ops[1]])) / 2
    
    skp = sum(1 for c in st[p] if c.value == 'Skip')
    
    w_my, w_op, w_sk = 5, 2, 3
    
    if strat == "offensive":
        w_my, w_op, w_sk = 7, 2, 2
    elif strat == "defensive":
        w_my, w_op, w_sk = 4, 4, 5
        
    return 50 - (w_my * my_c) + (w_op * op_c) + (w_sk * skp)
    
 # testing if game works           
game_deck = generate_deck()

test_state = {
    'Player1': [game_deck.pop() for _ in range(5)],
    'Player2': [game_deck.pop() for _ in range(5)],
    'Ai': [game_deck.pop() for _ in range(5)],
    'top_card': game_deck.pop(),
    'deck': game_deck
}

simulate_random_game(test_state)

starting game! top card: Green 8

--- Player1's turn ---
hand: [Yellow 1, Red 6, Blue 2, Green Skip, Red 8]
current score: 55.0
action: plays Red 8

--- Player2's turn ---
hand: [Red 3, Red 5, Red 0, Yellow 4, Green 1]
current score: 24.0
action: plays Red 0

--- Ai's turn ---
hand: [Yellow 2, Red 9, Yellow 7, Blue 5, Blue 1]
current score: 46.0
action: plays Red 9

--- Player1's turn ---
hand: [Yellow 1, Red 6, Blue 2, Green Skip]
current score: 55.0
action: plays Red 6

--- Player2's turn ---
hand: [Red 3, Red 5, Yellow 4, Green 1]
current score: 29.0
action: plays Red 3

--- Ai's turn ---
hand: [Yellow 2, Yellow 7, Blue 5, Blue 1]
current score: 46.0
action: draws a card

--- Player1's turn ---
hand: [Yellow 1, Blue 2, Green Skip]
current score: 59.0
action: draws a card

--- Player2's turn ---
hand: [Red 5, Yellow 4, Green 1]
current score: 38.0
action: plays Red 5

--- Ai's turn ---
hand: [Yellow 2, Yellow 7, Blue 5, Blue 1, Green 3]
current score: 42.0
action: plays Blue 5

--- P